# Fire VASE validation notebook

Run any section independently to reproduce one QA artifact. The default sample is real FIRED event `20657` with daily GridMET maximum temperature. Restore the materialized data lake under `data_lake/fire-vase-data-lake-v0.1/files` before running.

In [ ]:
from IPython.display import Image, display
from cubedynamics.validation import ValidationPaths

paths = ValidationPaths.discover()
FIRE_ID = 20657
VARIABLE = 'tmmx'

## Optional expected-failure contrast

Prove that the validators reject plausible mistakes: reversed latitude values, time planes attached to the wrong dates, a missing day, and a real FIRED event that exceeds the simplification-area threshold. These expected failures are kept separate from the six production checks.

In [ ]:
from cubedynamics.validation.contrast import run_contrast_suite

contrast_results, contrast_manifest = run_contrast_suite(
    paths,
    cube_fire_id=FIRE_ID,
    geometry_fire_id=72016,
    variable=VARIABLE,
    output_root=paths.repo_root / 'output/validation/contrast',
)
display({result.module: result.metrics for result in contrast_results})
display(Image(filename=contrast_results[0].artifacts['plot']))
display(Image(filename=contrast_results[1].artifacts['plot']))
contrast_manifest

## 1. Pipe grammar and lazy backend

Compare the direct CubeDynamics verb with the pipe form over the same chunked GridMET subset.

In [ ]:
from cubedynamics.validation.pipeline import run_pipeline_validation

pipeline = run_pipeline_validation(paths, fire_id=FIRE_ID, variable=VARIABLE)
display(pipeline.metrics)
display(Image(filename=pipeline.artifacts['plot']))

## 2. Real data cube and HTML integrity

Render every time slice and interior spatial plane from the real GridMET sample, then decode the HTML rasters and compare every pixel, coordinate, date, checksum, and corner landmark with the source cube.

In [ ]:
from IPython.display import IFrame
from cubedynamics.validation.cube import run_cube_validation

cube = run_cube_validation(paths, fire_id=FIRE_ID, variable=VARIABLE)
display(cube.metrics)
display(Image(filename=cube.artifacts['plot']))
display(IFrame(src=cube.artifacts['interactive_html'], width='100%', height=760))

## 3. FIRED polygon and hull sensitivity

Show the accepted 0–125 m simplification range alongside 500 m and 1000 m stress tests, then rebuild the directional time hull at each setting.

In [ ]:
from cubedynamics.validation.geometry import run_geometry_validation

geometry = run_geometry_validation(
    paths,
    fire_id=FIRE_ID,
    tolerances_m=(0, 125, 500, 1000),
    operational_max_tolerance_m=125,
    n_theta=96,
)
display(geometry.metrics)
display(Image(filename=geometry.artifacts['plot']))

## 4. Three-dimensional hull construction and averaging decisions

Show raw daily polygons, equal-step boundary samples, directional support rings, the triangulated production mesh, and 3-day, 7-day, and cumulative-envelope alternatives.

In [ ]:
from cubedynamics.validation.hull3d import run_hull3d_validation

hull3d = run_hull3d_validation(
    paths, fire_id=FIRE_ID, n_theta=96, averaging_windows=(1, 3, 7)
)
display(hull3d.metrics)
display(Image(filename=hull3d.artifacts['plot']))
display(IFrame(src=hull3d.artifacts['interactive_html'], width='100%', height=720))

## 5. GridMET date and polygon attribution

Recompute every lake-table centroid value from annual NetCDF and compare centroid, cell-center, and fractional pixel-overlap climate assignment.

In [ ]:
from cubedynamics.validation.climate import run_climate_validation

climate = run_climate_validation(paths, fire_id=FIRE_ID, variable=VARIABLE)
display(climate.metrics)
display(Image(filename=climate.artifacts['plot']))

## 6. External and upstream-source validation

The FIRED daily/event geometry check is offline. Set `EXTERNAL_NETWORK = True` to also query the independent NCAR/GDEX GridMET OPeNDAP mirror.

In [ ]:
from cubedynamics.validation.external import run_external_validation

EXTERNAL_NETWORK = False
external = run_external_validation(
    paths,
    fire_id=FIRE_ID,
    variable=VARIABLE,
    external_network=EXTERNAL_NETWORK,
)
display(external.metrics)
display(Image(filename=external.artifacts['plot']))

## Collate the current module results

The command-line runner normally creates the report. This cell shows the same collation step for results generated in this notebook.

In [ ]:
from cubedynamics.validation.report import build_validation_pdf

report = build_validation_pdf(
    [pipeline, cube, geometry, hull3d, climate, external],
    paths.repo_root / 'output/pdf/fire_vase_validation_report.pdf',
)
report